In [ ]:
# 1. czy problem dotyczy też innych partycjonowanych tabel?
for t in ['fact_work_orders', 'fact_sales_transactions', 'fact_invoices', 'fact_payments',
          'fact_inventory_movements', 'fact_appointments', 'fact_purchase_orders']:
    nulls = spark.sql(f"SELECT count(*) FROM car_workshop.fact.{t} WHERE year IS NULL").first()[0]
    total = spark.table(f'car_workshop.fact.{t}').count()
    print(f'{t}: {nulls:,}/{total:,} NULL year')

# 2. dowód, że pliki są OK (batch reader robi partition discovery sam):
spark.read.parquet('/Volumes/car_workshop/fact/fact_files/fact_sales_transactions') \
    .select('year', 'month').distinct().show(5)

In [ ]:
TRUNCATE TABLE car_workshop.fact.fact_sales_transactions;
TRUNCATE TABLE car_workshop.fact.fact_purchase_orders;


In [ ]:
dbutils.fs.rm('/Volumes/car_workshop/fact/autoloader_checkpoints/fact_sales_transactions/', recurse=True)
dbutils.fs.rm('/Volumes/car_workshop/fact/autoloader_checkpoints/fact_purchase_orders/', recurse=True)


In [ ]:
% run ./fake_car_workshop/autoloader.ipynb

In [ ]:
for t in ['fact_sales_transactions', 'fact_purchase_orders']:
    df = spark.table(f'car_workshop.fact.{t}')
    nulls = df.filter('year IS NULL').count()
    print(f'{t}: {df.count():,} rows, {nulls} NULL year')
    df.select('year').distinct().orderBy('year').show()
